In [5]:
import pandas as pd
import regex as re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV # Usaste esto para obtener model_2
from scipy.stats import loguniform


In [ ]:

# Instancia lematizador
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words('english'))

# 2. Recrear la función de limpieza (limpiar_url)

def limpiar_url(url):
    """Limpia una URL aplicando la misma lógica que en el entrenamiento."""
    # Eliminamos protocolos (http, https)
    url = re.sub(r'https?://', '', url)
    # Convertimos a minúsculas
    url = url.lower()
    # Eliminamos www.
    url = re.sub(r'www\.', '', url)
    # Eliminamos todo lo que no sean letras o números
    url = re.sub(r'[^a-zA-Z0-9\s]', ' ', url)
    # Tokenizamos y eliminamos stop words (y lematización)
    url = url.split()
    # Lematizamos y filtramos stopwords
    url = [lemmatizer.lemmatize(word) for word in url if word not in stop_words]
    # Revertimos a string
    url = ' '.join(url)
    return url

# 3. Cargar datos (necesario para ajustar el vectorizador)
# El notebook usa un DataFrame llamado 'df' que asumo tiene las columnas 'url' y 'label'.
# Reemplaza 'path_to_your_data.csv' con tu ruta real.
# df = pd.read_csv('path_to_your_data.csv') 
df = pd.read_csv('../data/raw/12 url_spam.csv', sep=',')
# # Placeholder de datos (NECESITAS REEMPLAZAR ESTO)
# Simulación de carga y preprocesamiento de tu notebook
data = {'url': ['google.com/search?q=legit', 'free-money-now.net/click', 'amazon.com/products/shoes', 'scam-site.biz/get-rich-fast'],
    'label': [0, 1, 0, 1]}

df = pd.DataFrame(data)
df['corpus'] = df['url'].apply(limpiar_url)

# 4. Vectorización (Ajustar y Transformar)
# Vectorizador usado en el notebook: TfidfVectorizer con max_features=5000
vectorizador = TfidfVectorizer(max_features=5000)
X = vectorizador.fit_transform(df['corpus']).toarray()
y = df.label

# 5. División de datos (Usaste 80% train / 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=18)

# 6. Recrear el modelo (model_2)
# Como no puedo recrear la RandomizedSearchCV, definiremos el SVC básico 
# y lo entrenaremos como si fuera el modelo final (model_2).
# Si tienes los hiperparámetros óptimos, úsalos aquí.
model_2 = SVC(kernel="linear", random_state=18)
# Nota: Si tu modelo final fue 'model_2 = random_search.best_estimator_', 
# este paso lo simula de forma simplificada.
model_2.fit(X_train, y_train) 
print("Configuración completada: El modelo (model_2) y el vectorizador están listos.")

Configuración completada: El modelo (model_2) y el vectorizador están listos.


In [7]:
def predecir_url_spam(url, modelo=model_2, vectorizador=vectorizador):
    """
    Clasifica una URL como SPAM o NO SPAM utilizando el modelo entrenado.

    Args:
        url (str): La URL a clasificar.
        modelo: El objeto SVC entrenado (model_2).
        vectorizador: El objeto TfidfVectorizer ajustado.

    Returns:
        str: El resultado de la predicción.
    """
    
    # 1. Preprocesamiento: Aplicar la misma limpieza que se usó en el entrenamiento
    url_limpia = limpiar_url(url)
    
    # 2. Vectorización: Transformar la URL limpia a un vector de características
    # Se pasa como una lista para que el vectorizador lo procese
    X_nueva = vectorizador.transform([url_limpia]).toarray()
    
    #     
    # 3. Predicción
    prediccion = modelo.predict(X_nueva)
    
    # 4. Interpretación del resultado
    # Asumo que 1 = SPAM y 0 = NO SPAM, según la convención típica de datasets de spam.
    if prediccion[0] == 1:
        return "⚠️ SPAM - La URL tiene características de un enlace malicioso."
    else:
        return "✅ NO SPAM - La URL parece legítima."

In [8]:
# URLs para probar
url_legitima = "https://www.wikipedia.org/wiki/Aprendizaje_automático"
url_spam = "http://www.free-money-now-no-survey.net/clickhere?id=123"

# Resultados
print(f"Predicción para '{url_legitima}':")
print(predecir_url_spam(url_legitima))

print("-" * 30)

print(f"Predicción para '{url_spam}':")
print(predecir_url_spam(url_spam))

Predicción para 'https://www.wikipedia.org/wiki/Aprendizaje_automático':
✅ NO SPAM - La URL parece legítima.
------------------------------
Predicción para 'http://www.free-money-now-no-survey.net/clickhere?id=123':
⚠️ SPAM - La URL tiene características de un enlace malicioso.


In [10]:
url_prueba = 'https://4geeks.com/es/syllabus/spain-ds-pt-20/project/NLP-project-tutorial'

# Resultados
print(f"Predicción para '{url_prueba}':")
print(predecir_url_spam(url_prueba))

Predicción para 'https://4geeks.com/es/syllabus/spain-ds-pt-20/project/NLP-project-tutorial':
✅ NO SPAM - La URL parece legítima.
